# Metronome Compass Testing

This notebook provides utilities for testing the complex `metronome_compass` (full battle tracking) by generating seeds that produce specific outcomes.

In [12]:
%load_ext autoreload
%autoreload 2
from claytonlib.safari import advance_rng
from claytonlib.metronome_compass_full import precompute_path, render_path
from claytonlib.moves import _moves_by_number, resolve_move
import datetime as dt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## RNG Reversing Utility

LCRNG: $state_{n+1} = (state_n \cdot 1103515245 + 24691) \pmod{2^{32}}$

Inverse: $state_n = ((state_{n+1} - 24691) \cdot 0xEEB9EB65) \pmod{2^{32}}$

In [13]:
def reverse_rng(state: int, n: int = 1) -> int:
    """Backtrack the LCRNG n steps."""
    inv_mult = 0xEEB9EB65  # Correct multiplicative inverse
    for _ in range(n):
        state = ((state - 24691) * inv_mult) & 0xFFFFFFFF
    return state

def generate_seed_for_move(move_name: str, magikarp_level: int = 2) -> int:
    """Generate an artificial seed that hits a specific Metronome move on Turn 1."""
    move = resolve_move(move_name)
    if not move or not move.metronome_usable:
        raise ValueError(f"Move {move_name!r} is not metronome-usable.")
    
    target_num = move.number
    pool = 467
    
    # We want (roll >> 16) % pool + 1 == target_num
    # So (roll >> 16) % pool == target_num - 1
    target_val = target_num - 1
    
    state = target_val << 16
    
    # Sequence to Turn 1 Metronome roll:
    # 6 advances (Battle Start)
    # 1 advance (Magikarp Select if level >= 15)
    # 4 advances (BeforeTurn)
    # 1 advance (Metronome roll)
    
    advances = 6 + (1 if magikarp_level >= 15 else 0) + 4 + 1
    
    seed = reverse_rng(state, advances)
    return seed

def verify_seed(seed: int, magikarp_level: int = 2):
    path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=10)
    print(f"Seed: 0x{seed:08X}")
    print(f"Path: {render_path(path)}")
    
    moves = _moves_by_number()
    move_num = path[0][0].move_num
    print(f"Move: {moves[move_num].name} (M{move_num:03d})")

## Example: Seed for Flamethrower

In [14]:
seed = generate_seed_for_move("Flamethrower")
verify_seed(seed)

Seed: 0xAE9FD1B7
Path: M053hKsp M132?_
Move: Flamethrower (M053)


## Example: Seed for Splash (Turn 1 Metronome)

In [19]:
seed = generate_seed_for_move("Splash")
verify_seed(seed)

Seed: 0x5C7CD1B7
Path: M150?_
Move: Splash (M150)


## Testing other moves

In [22]:
seed = generate_seed_for_move("Taunt")
verify_seed(seed)

Seed: 0x5417D1B7
Path: M269hSTR M282?_
Move: Taunt (M269)


## Multi-Turn Path Generation

In [16]:
seed = 0x12345678
path = precompute_path(seed, magikarp_level=15, opposite_gender=False, n_turns=5)
print(f"Seed: 0x{seed:08X} (Level 15 Magikarp)")
print(f"Path: {render_path(path)}")

Seed: 0x12345678 (Level 15 Magikarp)
Path: M126h~Ksp M063?_
